In [1]:
# Load feature-engineered train/val/test splits produced in 02_feat_engineer.ipynb
import pandas as pd

processed_dir = r"P:\AI\Project_Default\data\processed"

train_df = pd.read_parquet(f"{processed_dir}\\train_fe.parquet")
val_df = pd.read_parquet(f"{processed_dir}\\val_fe.parquet")
test_df = pd.read_parquet(f"{processed_dir}\\test_fe.parquet")

X_train, y_train = train_df.drop(columns=['target']), train_df['target']
X_val, y_val = val_df.drop(columns=['target']), val_df['target']
X_test, y_test = test_df.drop(columns=['target']), test_df['target']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(X_train.dtypes.value_counts())


Train: (964040, 76), Val: (206580, 76), Test: (206581, 76)
float64    60
str        11
int64       4
int32       1
Name: count, dtype: int64


In [2]:
# Drop date columns (signal already captured in credit_history_length) before building the pipeline
date_cols = ['issue_d', 'earliest_cr_line']
X_train = X_train.drop(columns=date_cols)
X_val = X_val.drop(columns=date_cols)
X_test = X_test.drop(columns=date_cols)

# Re-impute the 4 sentinel-filled columns (999 from 01_eda.ipynb Step 3) with a milder value
# for the linear model - the raw sentinel is far larger than any real observed value (e.g.
# mths_since_recent_inq real max is 25 vs sentinel 999), which distorts scaling and
# dominates logistic regression's coefficients. Keep the _missing_flag columns as-is;
# only replace the sentinel value itself, computed from X_train's non-sentinel max + 1.
# See notes/03_classifier.md for the full writeup of this bug.
sentinel_cols = ['mths_since_last_delinq', 'mths_since_recent_inq', 'num_tl_120dpd_2m', 'mo_sin_old_il_acct']
mild_sentinels = {}
for col in sentinel_cols:
    real_max = X_train.loc[X_train[col] != 999, col].max()
    mild_sentinels[col] = real_max + 1

for df in (X_train, X_val, X_test):
    for col, mild_value in mild_sentinels.items():
        df[col] = df[col].replace(999, mild_value)

print("Mild sentinel values (X_train non-missing max + 1):")
print(mild_sentinels)

# Preprocessing pipeline: ordinal encode known-ranked categoricals, one-hot the rest, scale numerics.
# All fit on X_train only via ColumnTransformer - applied to X_val/X_test without refitting.
# See notes/03_classifier.md for why ordinal vs one-hot was chosen per column.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

ordinal_cols = ['grade', 'sub_grade', 'emp_length']
ordinal_categories = [
    ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    ['A1','A2','A3','A4','A5','B1','B2','B3','B4','B5','C1','C2','C3','C4','C5',
     'D1','D2','D3','D4','D5','E1','E2','E3','E4','E5','F1','F2','F3','F4','F5',
     'G1','G2','G3','G4','G5'],
    ['unknown', '< 1 year', '1 year', '2 years', '3 years', '4 years', '5 years',
     '6 years', '7 years', '8 years', '9 years', '10+ years'],
]

onehot_cols = [
    'term', 'home_ownership', 'verification_status', 'purpose',
    'addr_state', 'application_type',
]

numeric_cols = [c for c in X_train.columns if c not in ordinal_cols + onehot_cols]

preprocessor = ColumnTransformer(transformers=[
    ('ordinal', OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ('scale', StandardScaler(), numeric_cols),
])

clf_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])

clf_pipeline.fit(X_train, y_train)
print("Baseline logistic regression trained.")


Mild sentinel values (X_train non-missing max + 1):
{'mths_since_last_delinq': np.float64(203.0), 'mths_since_recent_inq': np.float64(26.0), 'num_tl_120dpd_2m': np.float64(7.0), 'mo_sin_old_il_acct': np.float64(725.0)}
Baseline logistic regression trained.


In [3]:
# Peek at learned coefficients - map each back to its final feature name
feature_names = clf_pipeline.named_steps['preprocess'].get_feature_names_out()
coefs = clf_pipeline.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False)

print("Top 15 features by absolute coefficient magnitude:")
print(coef_df.head(15)[['feature', 'coefficient']].to_string(index=False))


Top 15 features by absolute coefficient magnitude:
                       feature  coefficient
       onehot__purpose_wedding    -0.519716
onehot__purpose_small_business     0.410546
         onehot__addr_state_WV    -0.400520
         onehot__addr_state_DC    -0.379660
         onehot__addr_state_MS     0.354994
         onehot__addr_state_OR    -0.351822
       onehot__term_ 36 months    -0.336805
         onehot__addr_state_VT    -0.328529
         onehot__addr_state_ME    -0.307193
         onehot__addr_state_NH    -0.279629
         onehot__addr_state_NE     0.271520
         onehot__addr_state_CO    -0.267591
         onehot__addr_state_AR     0.267078
         onehot__addr_state_OK     0.259319
         onehot__addr_state_WA    -0.258317


In [4]:
# Sanity check: sample sizes for the addr_state values dominating the top-15 coefficients
flagged_states = ['WV', 'DC', 'MS', 'OR', 'VT', 'ME', 'NH', 'NE', 'CO', 'AR', 'OK', 'WA']
state_counts = X_train['addr_state'].value_counts()
print(state_counts.loc[flagged_states].sort_values())
print()
print("For comparison, largest states by count:")
print(state_counts.head(5))
print()
print("Median state count:", state_counts.median())


addr_state
ME     1472
VT     1890
DC     2469
NE     2621
WV     3473
NH     4564
MS     4759
AR     7096
OK     8807
OR    11738
WA    20777
CO    20979
Name: count, dtype: int64

For comparison, largest states by count:
addr_state
CA    140777
TX     79086
NY     78796
FL     68672
IL     36941
Name: count, dtype: int64

Median state count: 11738.0


In [4]:
# Evaluate baseline logistic regression on the validation set
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix

val_proba = clf_pipeline.predict_proba(X_val)[:, 1]
val_pred = clf_pipeline.predict(X_val)

roc_auc = roc_auc_score(y_val, val_proba)
f1 = f1_score(y_val, val_pred)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"F1 (default class): {f1:.4f}")
print()
print("Classification report:")
print(classification_report(y_val, val_pred, target_names=['No Default', 'Default']))
print("Confusion matrix:")
print(confusion_matrix(y_val, val_pred))


ROC-AUC: 0.7178
F1 (default class): 0.4458

Classification report:
              precision    recall  f1-score   support

  No Default       0.88      0.66      0.76    163415
     Default       0.34      0.65      0.45     43165

    accuracy                           0.66    206580
   macro avg       0.61      0.66      0.60    206580
weighted avg       0.77      0.66      0.69    206580

Confusion matrix:
[[108629  54786]
 [ 15067  28098]]


In [5]:
# Tree-based comparison: Random Forest on the same train/val split.
# Trees split on thresholds rather than magnitude, so no scaling needed, and no need to
# one-hot the unordered categoricals - label/ordinal encoding is fine for a tree even on
# unordered categories, since the tree can carve out arbitrary threshold groups itself.
# Also: no sentinel re-imputation here - keep the original 999 sentinel + flag from
# 01_eda.ipynb (X_train/X_val were already mild-sentinel-fixed above for the linear model,
# so we reload fresh copies here to test the tree against the *original* 999 sentinel first).
# See notes/03_classifier.md for the full baseline writeup and what this comparison is for.
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder as OE

train_df_rf = pd.read_parquet(f"{processed_dir}\\train_fe.parquet")
val_df_rf = pd.read_parquet(f"{processed_dir}\\val_fe.parquet")

X_train_rf = train_df_rf.drop(columns=['target'] + date_cols)
X_val_rf = val_df_rf.drop(columns=['target'] + date_cols)
y_train_rf = train_df_rf['target']
y_val_rf = val_df_rf['target']

categorical_cols_rf = X_train_rf.select_dtypes(include='str').columns.tolist()

rf_preprocessor = ColumnTransformer(transformers=[
    ('encode', OE(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols_rf),
], remainder='passthrough')

rf_pipeline = Pipeline(steps=[
    ('preprocess', rf_preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200, max_depth=12, class_weight='balanced',
        n_jobs=-1, random_state=42,
    )),
])

rf_pipeline.fit(X_train_rf, y_train_rf)
print("Random Forest trained.")

rf_val_proba = rf_pipeline.predict_proba(X_val_rf)[:, 1]
rf_val_pred = rf_pipeline.predict(X_val_rf)

print(f"ROC-AUC: {roc_auc_score(y_val_rf, rf_val_proba):.4f}")
print(f"F1 (default class): {f1_score(y_val_rf, rf_val_pred):.4f}")
print()
print(classification_report(y_val_rf, rf_val_pred, target_names=['No Default', 'Default']))
print(confusion_matrix(y_val_rf, rf_val_pred))


Random Forest trained.
ROC-AUC: 0.7167
F1 (default class): 0.4438

              precision    recall  f1-score   support

  No Default       0.88      0.64      0.74    163415
     Default       0.33      0.67      0.44     43165

    accuracy                           0.65    206580
   macro avg       0.61      0.66      0.59    206580
weighted avg       0.77      0.65      0.68    206580

[[104782  58633]
 [ 14132  29033]]


In [6]:
# Hyperparameter tuning round 2: the first search's best params (n_estimators=200,
# min_samples_leaf=50, max_features=0.3, max_depth=16) all sat at the edge of the tested
# range - a sign the true optimum may lie outside it, not that we found a real peak.
# Widened the ranges past those edges (deeper trees, larger leaf sizes, more trees) while
# keeping n_iter/cv the same size as before to stay time-boxed after the earlier 100+
# minute run. See notes/03_classifier.md.
from sklearn.model_selection import RandomizedSearchCV

param_distributions_v2 = {
    'model__n_estimators': [200, 300, 400],
    'model__max_depth': [16, 20, 24, None],
    'model__min_samples_leaf': [50, 100, 200],
    'model__max_features': [0.3, 0.5, 0.7],
}

rf_search_v2 = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_distributions_v2,
    n_iter=6,
    scoring='roc_auc',
    cv=2,
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

rf_search_v2.fit(X_train_rf, y_train_rf)

print(f"Best CV ROC-AUC: {rf_search_v2.best_score_:.4f}")
print(f"Best params: {rf_search_v2.best_params_}")

best_rf_v2_val_proba = rf_search_v2.best_estimator_.predict_proba(X_val_rf)[:, 1]
best_rf_v2_val_pred = rf_search_v2.best_estimator_.predict(X_val_rf)

print(f"\nTuned RF v2 - Validation ROC-AUC: {roc_auc_score(y_val_rf, best_rf_v2_val_proba):.4f}")
print(f"Tuned RF v2 - Validation F1 (default class): {f1_score(y_val_rf, best_rf_v2_val_pred):.4f}")


Fitting 2 folds for each of 6 candidates, totalling 12 fits
Best CV ROC-AUC: 0.7196
Best params: {'model__n_estimators': 400, 'model__min_samples_leaf': 50, 'model__max_features': 0.3, 'model__max_depth': None}

Tuned RF v2 - Validation ROC-AUC: 0.7227
Tuned RF v2 - Validation F1 (default class): 0.4491
